# Decision Modeling Project


## Step 1:

### Formulation of the problem:

#### Variables, Sets, Data:
- $x_{i,j} \in \{0,1\}$ if brick $i$ is assigned to SR $j$
- $d_{i,j}$ = distance between brick $i$ and SR $j$ (data)
- $v_i$ = workload for brick $i$ (value between 0 and 1) (data)
- $S$ = set of SRs
- $B$ = set of bricks $(1, 2, 3, \ldots)$

#### Constraints (30 in total):

$\sum_{i \in B} x_{i,1} \cdot v_i < 1.2$

$\sum_{i \in B} x_{i,2} \cdot v_i < 1.2$

$\sum_{i \in B} x_{i,3} \cdot v_i < 1.2$

$\sum_{i \in B} x_{i,4} \cdot v_i < 1.2$

$\sum_{i \in B} x_{i,1} \cdot v_i > 0.8$

$\sum_{i \in B} x_{i,2} \cdot v_i > 0.8$

$\sum_{i \in B} x_{i,3} \cdot v_i > 0.8$

$\sum_{i \in B} x_{i,4} \cdot v_i > 0.8$

22 times this constraints for each brick i:

$\sum_{j \in S} x_{1,j} = 1$ (example with i=1)

#### Objective Function:

$$\min \sum_{i \in B} \sum_{j \in S} d_{i,j} \cdot x_{i,j}$$

$$\min \sum_{i \in B} \sum_{j \in S} \left({x'}_{i,j} - 2{x'}_{i,j} \cdot x_{i,j} + x_{i,j}\right)$$

### Using GUROBI for 22 Bricks and 4 SR

In [ ]:
!pip install gurobipy


In [3]:
import gurobipy
from gurobipy import Model, GRB
import matplotlib.pyplot as plt

#  implement your two mono-objective models using GUROBI, and solve the instance with 22 Bricks
# and 4 Sales Representatives

model = Model("Bricks_and_Sales_Reps")
num_bricks = 22
num_SR = 4

# --- Data ---
distances = [[16.16, 24.08, 24.32, 21.12], 
     [19, 26.47, 27.24, 17.33], 
     [25.29, 32.49, 33.42, 12.25], 
     [0, 7.93, 8.31, 36.12], 
     [3.07, 6.44, 7.56, 37.37], 
     [1.22, 7.51, 8.19, 36.29], 
     [2.80, 10.31, 10.95, 33.5], 
     [2.87, 5.07, 5.67, 38.8], 
     [3.8, 8.01, 7.41, 38.16], 
     [12.35, 4.52, 4.35, 48.27], 
     [11.11, 3.48, 2.97, 47.14], 
     [21.99, 22.02, 24.07, 39.86], 
     [8.82, 3.3, 5.36, 43.31], 
     [7.93, 0, 2.07, 43.75], 
     [9.34, 2.25, 1.11, 45.43], 
     [8.31, 2.07, 0, 44.43], 
     [7.31, 2.44, 1.11, 43.43], 
     [7.55, 0.75, 1.53, 43.52], 
     [11.13, 18.41, 19.26, 25.4], 
     [17.49, 23.44, 24.76, 23.21], 
     [11.03, 18.93, 19.28, 25.43], 
     [36.12, 43.75, 44.43, 0]] 

workloads = [0.1609, 0.1164, 0.1026, 0.1516, 0.0939, 
     0.1320, 0.0687, 0.0930, 0.2116, 0.2529, 
     0.0868, 0.0828, 0.0975, 0.8177, 0.4115, 
     0.3795, 0.0710, 0.0427, 0.1043, 0.0997, 
     0.1698, 0.2531]


# je veux formaliser les deux objectifs suivants :
# 1. Minimiser le cout total de distribution des briques
# 2. Maximiser la valeur totale des briques distribuées

# --- Variables ---
x = model.addVars(num_bricks, num_SR, vtype=GRB.BINARY, name="x")

# --- Constraints ---
for j in range(num_SR):
    model.addConstr(sum(x[i,j] * workloads[i] for i in range(num_bricks)) <= 1.2, name=f"max_workload_SR{j+1}")
    model.addConstr(sum(x[i,j] * workloads[i] for i in range(num_bricks)) >= 0.8, name=f"min_workload_SR{j+1}")

for i in range(num_bricks):
    model.addConstr(sum(x[i,j] for j in range(num_SR)) == 1, name=f"brick_{i+1}_assignment")

# --- Objective function 1 : Minimizing the distance ---
model.setObjective(sum(distances[i][j] * x[i,j] for i in range(num_bricks) for j in range(num_SR)),GRB.MINIMIZE)

# --- Objective function 2 : Minimize the disruption ---
# ...

Restricted license - for non-production use only - expires 2026-11-23


In [6]:
model.optimize()


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 7735U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 30 rows, 88 columns and 264 nonzeros
Model fingerprint: 0x4cc1d546
Variable types: 0 continuous, 88 integer (88 binary)
Coefficient statistics:
  Matrix range     [4e-02, 1e+00]
  Objective range  [8e-01, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-01, 1e+00]
Presolved: 30 rows, 88 columns, 264 nonzeros

Continuing optimization...


Explored 1 nodes (25 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 16 (of 16 available processors)

Solution count 3: 154.6 221.68 338.3 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.546000000000e+02, best bound 1.546000000000e+02, gap 0.0000%


In [10]:
for j in range(num_SR):
    assigned = [i+1 for i in range(num_bricks) if x[i,j].X > 0.5]
    print(f"SR {j+1}: {assigned}")

SR 1: [4, 5, 6, 7, 8, 9, 12, 19, 20]
SR 2: [11, 13, 14, 18]
SR 3: [10, 15, 16, 17]
SR 4: [1, 2, 3, 21, 22]
